# Build the global panel

**Purpose.** This notebook builds the country-year panel of international tourism indicators from the raw World Bank files, checks it, and reports how much data each economy has. The panel is the input of every later notebook. The notebook also records which economies have a complete series of the outcome, international tourism receipts as a percentage of GDP, over the estimation window, because only such economies can serve as donors in the synthetic-control analysis. The loader stops with an error that names the file or lists the economies when an indicator file has no data row, when an economy appears in the files of two groups, or when the economies of the files differ from those of `country_metadata.csv`. The range check stops with an error that lists the offending economy-years, indicators and values.

**Input files.** The indicator files `wdi_{group}_{indicator}.csv`, one for each group of economies and each of the indicators ST.INT.RCPT.CD, ST.INT.ARVL, NY.GDP.MKTP.CD, SP.POP.TOTL and IS.AIR.PSGR, and the file `country_metadata.csv`, all in the raw data directory. The first code cell prints the directory.

**Output files.** `global_panel.csv` in the processed data directory; `panel_coverage.csv` and `donor_eligibility.csv` in the results directory; the figures `outcome_coverage_heatmap.png` and `outcome_by_year.png` in the figures directory; and `figure_inventory_00_build_global_panel.csv` in the results directory.

**How to run.** On the desktop that holds the real data, set `DTT_RUN_REAL=1` and execute all cells, for example with `jupyter nbconvert --to notebook --execute --output-dir executed notebooks/00_build_global_panel.ipynb`, which writes the executed copy to the folder `executed` and leaves the notebook itself unchanged. To test the notebook on a simulated world, set `DTT_WORLD=sim` together with `DTT_SIM_DIR`, `DTT_RESULTS_DIR` and `DTT_FIGURE_DIR`, each pointing to a directory outside the repository. A simulated run refuses an output directory inside the data, results or figures directory of the repository, never reads the real files and never writes to the results directory of the repository.

**Run time.** Well under one minute in both modes.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "src" / "dtt" / "__init__.py").is_file())
sys.path.insert(0, str(_root / "src"))

from dtt import workflow

ctx = workflow.setup_run("00_build_global_panel")
ctx.describe()

## Settings

The notebook has no setting that depends on the mode. The outcome, the estimation window used for the donor table, the pandemic years that the figures mark and the largest plausible value of the outcome are set here once.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from dtt.panel import (
    GROUPS,
    INDICATORS,
    OUTCOME_PLAUSIBLE_MAX,
    PANEL_COLUMNS,
    RANGE_ROWS_LISTED,
    RAW_COLUMNS,
    build_global_panel,
    check_value_ranges,
    coverage_report,
    load_wdi,
    read_panel,
    write_panel,
)

OUTCOME = "receipts_pct_gdp"
WINDOW_FIRST_YEAR, WINDOW_LAST_YEAR = 1995, 2019
PANDEMIC_YEARS = (2020, 2021, 2022)

print(f"Outcome: {OUTCOME} (international tourism receipts, percent of GDP)")
print(f"Estimation window of the donor table: {WINDOW_FIRST_YEAR} to {WINDOW_LAST_YEAR}")
print(f"Pandemic years marked in the figures: {PANDEMIC_YEARS[0]} to {PANDEMIC_YEARS[-1]}")
print(f"Plausible range of the outcome: 0 to {OUTCOME_PLAUSIBLE_MAX:.0f} percent of GDP")
print(f"Offending rows listed by the range check: the first {RANGE_ROWS_LISTED}")
print(f"Target economy: {ctx.target_name} ({ctx.target_iso3})")


def display_path(path: Path) -> str:
    """Return a path relative to the repository root, or the file name for a path outside it.

    Parameters
    ----------
    path : pathlib.Path
        Path of a file.

    Returns
    -------
    str
        The path as it is printed in this notebook.
    """
    try:
        return str(Path(path).resolve().relative_to(ctx.root))
    except ValueError:
        return Path(path).name

## Build and store the panel

The function `build_global_panel` validates the indicator files before it builds anything. Each file must have the columns `iso3`, `year` and `value`, at least one data row, integer years, numeric or empty values and no repeated economy and year. An economy may appear in the files of one group only, and the economies of the files must be exactly the economies of `country_metadata.csv`. A failed check raises an error that names the file or lists the economies. The function then arranges one row for every economy and every year of the files and adds the derived ratios and logarithms. Missing values stay missing: nothing is imputed, interpolated or extrapolated. The panel is written with `write_panel`, stamped like the result tables so that the later notebooks detect a panel that changed after this notebook wrote it, and read back to confirm that the stored file reproduces the panel in memory exactly. The cell ends by repeating the file checks on the raw files, so that their results are printed: the number of data rows of each file, the number of groups of each economy and the comparison of the economies with the metadata.

In [ ]:
panel = build_global_panel(ctx.raw_dir)
panel_path = write_panel(panel, ctx.processed_dir / "global_panel.csv")
ctx.stamp_file(panel_path)
pd.testing.assert_frame_equal(read_panel(panel_path), panel, check_dtype=False, check_exact=True)

years = np.sort(panel["year"].unique())
economies = np.sort(panel["iso3"].unique())
print(f"Panel written to {display_path(panel_path)}")
print(
    f"The panel has {len(panel)} rows and {panel.shape[1]} columns for {economies.size} economies "
    f"and the years {years.min()} to {years.max()}."
)
print("Read back from the file: identical to the panel in memory.")

economy_table = panel.drop_duplicates("iso3")
print("\nEconomies by income level:")
print(economy_table["income_level"].value_counts(dropna=False).rename("economies").to_string())
print("\nEconomies by region:")
print(economy_table["region"].value_counts(dropna=False).rename("economies").to_string())

# The file checks of the loader, repeated on the raw files so that their results are printed.
long = load_wdi(ctx.raw_dir)
file_rows = long.groupby(["group", "indicator"]).size()
file_values = long.groupby(["group", "indicator"])["value"].count()
groups_of_economy = long.groupby("iso3")["group"].nunique()
economies_by_group = long.drop_duplicates("iso3").groupby("group")["iso3"].size()
metadata = pd.read_csv(ctx.raw_dir / "country_metadata.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
metadata_economies = set(metadata["iso3"].str.strip())
file_economies = set(long["iso3"])

print(f"\nRaw indicator files read: {file_rows.size} of {len(GROUPS) * len(INDICATORS)} expected.")
print(f"Data rows per file: minimum {int(file_rows.min())}, maximum {int(file_rows.max())}; rows with a value: minimum {int(file_values.min())}, maximum {int(file_values.max())}.")
print("Economies per group: " + ", ".join(f"{group} {int(n)}" for group, n in economies_by_group.items()))
print(f"Largest number of groups in which one economy appears: {int(groups_of_economy.max())}.")
print(
    f"Economies in the files: {len(file_economies)}; in country_metadata.csv: {len(metadata_economies)}; "
    f"in both: {len(file_economies & metadata_economies)}; in the panel: {economies.size}."
)

assert file_rows.size == len(GROUPS) * len(INDICATORS) and (file_rows > 0).all(), "an indicator file has no data row"
assert (groups_of_economy == 1).all(), "an economy appears in the files of more than one group"
assert file_economies == metadata_economies == set(economies), "the economies of the files differ from those of the metadata"
print("\nLoader checks passed: every indicator file has data rows, every economy is in one group, and the economies of the files are those of the metadata.")

## Data coverage

The coverage report counts, for every economy and each raw indicator, the years with a value (`n_years`), the first and last such year, the holes (`n_holes`) and the negative values (`n_negative`). A hole is a year between the first and the last year with a value that has no value itself. The outcome itself, which is derived from receipts and GDP, is summarised in the same way. The full report is saved as `panel_coverage.csv`. The table below summarises it by indicator. The checks that follow confirm that no economy-year row is repeated, that the counts agree with a count made directly on the panel, and that no value lies outside its plausible range. The function `check_value_ranges` flags a negative value of receipts, arrivals or air passengers, a GDP or population that is not positive, and an outcome above the largest plausible value of the settings cell. When it finds such values it stops with an error that gives their total number and lists the first offending rows with economy, year, indicator and value.

In [ ]:
assert not panel.duplicated(["iso3", "year"]).any(), "the panel repeats an economy and year"
coverage = coverage_report(panel)
ctx.save_table(coverage, "panel_coverage.csv")

by_indicator = coverage.groupby("indicator", sort=False).agg(
    economies_with_data=("n_years", lambda s: int((s > 0).sum())),
    economies_complete=("n_years", lambda s: int((s == years.size).sum())),
    median_years_observed=("n_years", "median"),
    earliest_first_year=("first_year", "min"),
    latest_last_year=("last_year", "max"),
    economies_with_holes=("n_holes", lambda s: int((s > 0).sum())),
    holes=("n_holes", "sum"),
    negative_values=("n_negative", "sum"),
)
print(f"Coverage of the {len(RAW_COLUMNS)} raw indicators over {years.size} panel years and {economies.size} economies")
print(by_indicator.to_string())

total_holes = int(coverage["n_holes"].sum())
pairs_with_holes = int((coverage["n_holes"] > 0).sum())
economies_with_holes = coverage.loc[coverage["n_holes"] > 0, "iso3"].nunique()
print(
    f"\nHoles, that is years without a value inside the observed span of an indicator: {total_holes}, "
    f"in {pairs_with_holes} of {len(coverage)} economy and indicator pairs and in {economies_with_holes} of {economies.size} economies."
)
outcome_coverage = coverage_report(panel, [OUTCOME])
outcome_in_window = panel.loc[panel["year"].between(WINDOW_FIRST_YEAR, WINDOW_LAST_YEAR), ["iso3", OUTCOME]]
print(
    f"Outcome {OUTCOME}: holes {int(outcome_coverage['n_holes'].sum())} in {int((outcome_coverage['n_holes'] > 0).sum())} economies, "
    f"negative values {int(outcome_coverage['n_negative'].sum())}; economy-years from {WINDOW_FIRST_YEAR} to {WINDOW_LAST_YEAR} without a value: "
    f"{int(outcome_in_window[OUTCOME].isna().sum())} of {len(outcome_in_window)}."
)
if total_holes:
    worst = coverage.sort_values(["n_holes", "iso3", "indicator"], ascending=[False, True, True]).head(5)
    print("\nLargest numbers of holes:")
    print(worst.to_string(index=False))

# Independent count of holes and negative values, made directly on the panel.
direct_holes = {}
direct_negative = {}
for column in RAW_COLUMNS:
    matrix = panel.pivot(index="iso3", columns="year", values=column).reindex(index=economies, columns=years).to_numpy(dtype=float)
    seen = np.isfinite(matrix)
    any_seen = seen.any(axis=1)
    first = np.argmax(seen, axis=1)
    last = years.size - 1 - np.argmax(seen[:, ::-1], axis=1)
    holes_here = np.where(any_seen, last - first + 1 - seen.sum(axis=1), 0)
    direct_holes.update({(iso3, column): int(h) for iso3, h in zip(economies, holes_here, strict=True)})
    negative_here = (np.where(seen, matrix, 0.0) < 0.0).sum(axis=1)
    direct_negative.update({(iso3, column): int(k) for iso3, k in zip(economies, negative_here, strict=True)})
reported_holes = {(r.iso3, r.indicator): int(r.n_holes) for r in coverage.itertuples()}
reported_negative = {(r.iso3, r.indicator): int(r.n_negative) for r in coverage.itertuples()}

assert len(coverage) == economies.size * len(RAW_COLUMNS)
assert coverage["n_years"].between(0, years.size).all()
assert coverage["n_holes"].ge(0).all() and (coverage["n_holes"] + coverage["n_years"] <= years.size).all()
assert reported_holes == direct_holes, "n_holes differs from the count made directly on the panel"
assert reported_negative == direct_negative, "n_negative differs from the count made directly on the panel"
check_value_ranges(panel, OUTCOME, OUTCOME_PLAUSIBLE_MAX)
print(
    "\nChecks passed: no economy-year row is repeated, the coverage report has one row per economy and indicator, "
    f"its {total_holes} holes and its negative values agree with a direct count, and no value is outside its plausible range."
)

## Donor eligibility

A donor needs an observed outcome in every year of the estimation window. The table records, for each economy, the number of years with an observed outcome in the window and whether the series is complete. A complete economy is not automatically a donor for every case, because notebook 02 also excludes economies that have an opening of their own close to the case.

In [ ]:
n_window_years = WINDOW_LAST_YEAR - WINDOW_FIRST_YEAR + 1
in_window = panel["year"].between(WINDOW_FIRST_YEAR, WINDOW_LAST_YEAR)
observed_flag = np.isfinite(panel.loc[in_window, OUTCOME].to_numpy(dtype=float))
counts = (
    pd.Series(observed_flag, index=panel.loc[in_window, "iso3"].to_numpy())
    .groupby(level=0)
    .sum()
    .astype(int)
    .reindex(economies, fill_value=0)
)
donor_eligibility = pd.DataFrame(
    {
        "iso3": counts.index.to_numpy(),
        f"n_years_outcome_{WINDOW_FIRST_YEAR}_{WINDOW_LAST_YEAR}": counts.to_numpy(),
        f"complete_{WINDOW_FIRST_YEAR}_{WINDOW_LAST_YEAR}": counts.to_numpy() == n_window_years,
    }
)
ctx.save_table(donor_eligibility, "donor_eligibility.csv")

n_col = f"n_years_outcome_{WINDOW_FIRST_YEAR}_{WINDOW_LAST_YEAR}"
complete_col = f"complete_{WINDOW_FIRST_YEAR}_{WINDOW_LAST_YEAR}"
n_complete = int(donor_eligibility[complete_col].sum())
print(
    f"{n_complete} of {len(donor_eligibility)} economies have an observed outcome in all {n_window_years} years "
    f"from {WINDOW_FIRST_YEAR} to {WINDOW_LAST_YEAR}."
)
incomplete = donor_eligibility.loc[~donor_eligibility[complete_col]]
if len(incomplete):
    print(f"\nNumber of observed years among the {len(incomplete)} incomplete economies:")
    print(incomplete[n_col].value_counts().sort_index().rename("economies").rename_axis("observed years").to_string())
    print("\nIncomplete economies with the fewest observed years:")
    print(incomplete.sort_values([n_col, "iso3"]).head(10).to_string(index=False))

## Sanity checks

The checks confirm that every economy has exactly one row for every year, that each derived column equals its definition recomputed from the raw columns, that a derived value is missing exactly where one of its inputs is missing or a denominator is zero, and that the raw indicators and the outcome lie in plausible ranges. The range check is the function `check_value_ranges` of the coverage section, applied to the panel in memory.

In [ ]:
def ratio(numerator: str, denominator: str, scale: float = 1.0) -> np.ndarray:
    """Return scale * numerator / denominator for two panel columns.

    Parameters
    ----------
    numerator, denominator : str
        Names of panel columns.
    scale : float, default 1.0
        Factor applied to the ratio.

    Returns
    -------
    numpy.ndarray
        The ratio, missing where an input is missing or the denominator is zero.
    """
    a = panel[numerator].to_numpy(dtype=float)
    b = panel[denominator].to_numpy(dtype=float)
    ok = np.isfinite(a) & np.isfinite(b) & (b != 0.0)
    out = np.full(a.shape, np.nan)
    out[ok] = scale * a[ok] / b[ok]
    return out


def log_of(column: str) -> np.ndarray:
    """Return the natural logarithm of a panel column.

    Parameters
    ----------
    column : str
        Name of a panel column.

    Returns
    -------
    numpy.ndarray
        The logarithm, missing where the value is missing or not positive.
    """
    x = panel[column].to_numpy(dtype=float)
    ok = np.isfinite(x) & (x > 0.0)
    out = np.full(x.shape, np.nan)
    out[ok] = np.log(x[ok])
    return out


# 1. One row per economy and year, in the documented column order.
assert list(panel.columns) == list(PANEL_COLUMNS)
assert not panel.duplicated(["iso3", "year"]).any()
rows_per_economy = panel.groupby("iso3").size()
assert (rows_per_economy == years.size).all()
assert panel.groupby("iso3")["year"].apply(lambda s: bool(np.array_equal(np.sort(s.to_numpy()), years))).all()
print(f"Rows: {economies.size} economies x {years.size} years = {len(panel)} rows, one per economy and year.")

# 2. Derived columns equal their definitions.
definitions = [
    ("receipts_pct_gdp", "100 * receipts_usd / gdp_usd", ratio("receipts_usd", "gdp_usd", 100.0)),
    ("log_receipts", "log(receipts_usd)", log_of("receipts_usd")),
    ("gdp_per_capita_usd", "gdp_usd / pop", ratio("gdp_usd", "pop")),
    ("log_gdp_pc", "log(gdp_per_capita_usd)", log_of("gdp_per_capita_usd")),
    ("log_pop", "log(pop)", log_of("pop")),
    ("log_gdp", "log(gdp_usd)", log_of("gdp_usd")),
    ("arrivals_per_capita", "arrivals / pop", ratio("arrivals", "pop")),
    ("receipts_per_arrival_usd", "receipts_usd / arrivals", ratio("receipts_usd", "arrivals")),
    ("log_receipts_per_arrival", "log(receipts_per_arrival_usd)", log_of("receipts_per_arrival_usd")),
    ("air_pax_per_capita", "air_pax / pop", ratio("air_pax", "pop")),
]
identity_rows = []
for column, formula, expected in definitions:
    actual = panel[column].to_numpy(dtype=float)
    same_missing = bool(np.array_equal(np.isnan(actual), np.isnan(expected)))
    defined = np.isfinite(expected)
    gap = np.abs(actual[defined] - expected[defined]) / np.maximum(np.abs(expected[defined]), 1e-300)
    worst = float(gap.max()) if gap.size else 0.0
    identity_rows.append(
        {
            "column": column,
            "definition": formula,
            "values_defined": int(defined.sum()),
            "values_missing": int((~defined).sum()),
            "missing_pattern_matches": same_missing,
            "largest_relative_difference": worst,
        }
    )
    assert same_missing, f"{column}: missing values do not match the definition"
    assert worst < 1e-12, f"{column}: differs from its definition by {worst:.2e}"
print("\nDerived columns recomputed from the raw columns:")
print(pd.DataFrame(identity_rows).to_string(index=False, formatters={"largest_relative_difference": "{:.1e}".format}))

# 3. Ranges of the raw indicators and of the outcome.
check_value_ranges(panel, OUTCOME, OUTCOME_PLAUSIBLE_MAX)
outcome_values = panel[OUTCOME].to_numpy(dtype=float)
finite_outcome = outcome_values[np.isfinite(outcome_values)]
assert finite_outcome.size > 0
above_gdp = panel.loc[panel[OUTCOME] > 100.0, ["iso3", "year", OUTCOME]]
largest = panel.loc[panel[OUTCOME].idxmax(), ["iso3", "year", OUTCOME]]
quantiles = np.percentile(finite_outcome, [0, 1, 25, 50, 75, 99, 100])
print(
    f"\nOutcome {OUTCOME}: {finite_outcome.size} observed economy-years, "
    f"{outcome_values.size - finite_outcome.size} missing."
)
print(
    "Minimum, 1st, 25th, 50th, 75th, 99th percentile and maximum: "
    + ", ".join(f"{q:.2f}" for q in quantiles)
)
print(f"Largest value: {largest[OUTCOME]:.2f} percent of GDP ({largest['iso3']}, {int(largest['year'])}).")
print(
    f"Observations above 100 percent of GDP: {len(above_gdp)}"
    + (f" in {above_gdp['iso3'].nunique()} economies, for example {', '.join(sorted(above_gdp['iso3'].unique())[:5])}." if len(above_gdp) else ".")
)

missing_metadata = economy_table[["name", "region", "income_level"]].isna().sum()
print("\nEconomies with missing metadata: " + ", ".join(f"{k} {int(v)}" for k, v in missing_metadata.items()))
print(
    "\nChecks passed: every economy has one row for every year, each derived column equals its definition, "
    f"and no value is outside its plausible range (outcome from 0 to {OUTCOME_PLAUSIBLE_MAX:.0f} percent of GDP)."
)

## Figure: coverage of the outcome

Each row is an economy and each column a year. A dark cell means that the outcome is observed and a white cell that it is missing. The economies are sorted by the number of years with an observed outcome, the largest number at the top. Observed cells in the pandemic years are drawn in a lighter shade and the dashed vertical lines enclose those years. The bars on the right give the number of observed years of each economy.

In [ ]:
wide = panel.pivot(index="iso3", columns="year", values=OUTCOME).reindex(columns=years)
observed = np.isfinite(wide.to_numpy(dtype=float))
n_observed = observed.sum(axis=1)
row_order = np.lexsort((wide.index.to_numpy(dtype=str), -n_observed))
pandemic_columns = np.isin(years, PANDEMIC_YEARS)
state = observed.astype(int)
state[:, pandemic_columns] *= 2
state = state[row_order]
row_labels = wide.index.to_numpy(dtype=str)[row_order]
n_rows = state.shape[0]

fig = plt.figure(figsize=(9.0, max(5.0, 0.135 * n_rows + 2.6)))
grid = fig.add_gridspec(1, 2, width_ratios=[6.0, 1.1], wspace=0.04)
ax = fig.add_subplot(grid[0, 0])
ax_bar = fig.add_subplot(grid[0, 1], sharey=ax)
ax.imshow(state, aspect="auto", cmap=ListedColormap(["white", "0.3", "0.7"]), vmin=0, vmax=2, interpolation="nearest")
ax.grid(False)
ax.set_yticks(np.arange(n_rows))
ax.set_yticklabels(row_labels, fontsize=5.5)
tick_positions = [i for i, year in enumerate(years) if year % 5 == 0]
ax.set_xticks(tick_positions)
ax.set_xticklabels([str(years[i]) for i in tick_positions])
first_p, last_p = np.flatnonzero(pandemic_columns)[[0, -1]]
for edge in (first_p - 0.5, last_p + 0.5):
    ax.axvline(edge, color="0.0", linestyle="--", linewidth=0.9)
ax.text((first_p + last_p) / 2, 1.004, "Pandemic years", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Year")
ax.set_ylabel("Economy (ISO3 code), sorted by number of observed years")
ax.set_title("Coverage of international tourism receipts (percent of GDP)", pad=18)
ax.legend(
    handles=[
        Patch(facecolor="0.3", edgecolor="0.15", label="Observed"),
        Patch(facecolor="0.7", edgecolor="0.15", label=f"Observed in the pandemic years {PANDEMIC_YEARS[0]} to {PANDEMIC_YEARS[-1]}"),
        Patch(facecolor="white", edgecolor="0.15", label="Missing"),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.62 / (fig.get_figheight() * ax.get_position().height)),
    ncol=3,
    fontsize=8,
)
ax_bar.barh(np.arange(n_rows), n_observed[row_order], color="0.5", height=0.8)
ax_bar.set_xlim(0, years.size)
ax_bar.set_xticks(np.arange(0, years.size + 1, 10))
ax_bar.tick_params(labelleft=False)
ax_bar.set_xlabel("Years observed (count)")
ax_bar.set_title("Observed years", pad=18)
ctx.save_figure(fig, "outcome_coverage_heatmap.png")

print(f"Economies with the outcome observed in every year: {int((n_observed == years.size).sum())} of {n_rows}.")
print(f"Economies with no observation of the outcome: {int((n_observed == 0).sum())}.")
pandemic_obs = observed[:, pandemic_columns].sum(axis=0)
print("Economies with the outcome observed, by pandemic year: " + ", ".join(f"{y} {int(n)}" for y, n in zip(years[pandemic_columns], pandemic_obs, strict=True)))

## Figure: the outcome over time

The shaded band spans the interquartile range of the outcome across all economies observed in a year, the dashed line is the median, and the solid line is the target economy. The pandemic years are shaded in grey. The lower panel counts the economies behind each year, because the set of economies changes where the coverage figure shows gaps.

In [ ]:
by_year = panel.groupby("year")[OUTCOME]
band = by_year.quantile([0.25, 0.5, 0.75]).unstack().reindex(years)
n_by_year = by_year.count().reindex(years)
target_series = panel.loc[panel["iso3"] == ctx.target_iso3].set_index("year")[OUTCOME].reindex(years)

fig, (ax, ax_n) = plt.subplots(
    2, 1, figsize=(9.0, 6.2), sharex=True, gridspec_kw={"height_ratios": [3.2, 1.0], "hspace": 0.1}
)
for a in (ax, ax_n):
    a.axvspan(PANDEMIC_YEARS[0] - 0.5, PANDEMIC_YEARS[-1] + 0.5, facecolor="0.0", alpha=0.16, edgecolor="none", zorder=4)
ax.fill_between(years, band[0.25], band[0.75], color="0.7", zorder=1, label="25th to 75th percentile of all economies")
ax.plot(years, band[0.5], color="0.25", linestyle="--", linewidth=1.6, zorder=2, label="Median of all economies")
ax.plot(years, target_series, color="0.0", linestyle="-", marker="o", markersize=3.5, linewidth=1.6, zorder=3, label=f"Target economy: {ctx.target_name}")
handles, labels = ax.get_legend_handles_labels()
handles.append(Patch(facecolor="0.84", edgecolor="0.5", label=f"Pandemic years {PANDEMIC_YEARS[0]} to {PANDEMIC_YEARS[-1]}"))
ax.set_ylabel("Tourism receipts (percent of GDP)")
ax.set_title("Outcome across economies and for the target economy")
ax.legend(handles=handles, loc="upper left", fontsize=8)
ax_n.bar(years, n_by_year, color="0.55", width=0.8, zorder=2)
ax_n.set_ylabel("Economies with data (count)")
ax_n.set_xlabel("Year")
ax_n.set_xticks([y for y in years if y % 5 == 0])
ctx.save_figure(fig, "outcome_by_year.png")

shown_years = sorted({int(years.min()), WINDOW_LAST_YEAR, PANDEMIC_YEARS[-1], int(years.max())} & set(int(y) for y in years))
summary = pd.DataFrame(
    {
        "economies": n_by_year.loc[shown_years].astype(int),
        "25th percentile": band.loc[shown_years, 0.25],
        "median": band.loc[shown_years, 0.5],
        "75th percentile": band.loc[shown_years, 0.75],
        ctx.target_iso3: target_series.loc[shown_years],
    }
)
print("Outcome (percent of GDP) in selected years:")
print(summary.round(2).to_string())

## Scope and limits

The panel holds the World Bank figures as delivered. International tourism receipts and GDP are both in current US dollars, so the outcome moves with exchange rates and price levels as well as with real activity. Coverage differs widely across economies. The donor table records only whether the outcome series is complete over the estimation window; it does not judge whether an economy is comparable to a host economy. The pandemic years are shown for reference, but they lie outside the estimation window, because the pandemic changed tourism in all economies at once. The file checks and the range checks find missing files, empty files, inconsistent sets of economies and implausible values, but they cannot find a value that is wrong and plausible. The summaries in this notebook describe the data and are not estimates of any effect.